# 79 — Search + Extraction Pipeline (Haystack)

## Goal
Retrieve documents, **extract structured fields**, and return
results as **JSON** — a common pattern for data ingestion
and information extraction.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Information extraction** | Pull structured fields from unstructured text |
| **Prompt engineering** | Design extraction prompts that produce JSON |
| **Pipeline chaining** | Retrieval → Extraction → Validation |
| **Error handling** | Graceful JSON parsing with fallbacks |

## Stack
- `haystack-ai` 2.27+
- `ollama-haystack` (OllamaChatGenerator)
- `InMemoryDocumentStore` + BM25 retrieval

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack.dataclasses import ChatMessage

print("All imports OK")

## Step 1 — Sample Documents

We simulate a collection of **job postings**.
Our goal: extract structured data (title, company, location,
salary, skills) from each.

In [ ]:
job_docs = [
    Document(
        content="""Senior Machine Learning Engineer at TechCorp
Location: San Francisco, CA (Hybrid)
Salary: $180,000 - $220,000/year
We are looking for an experienced ML engineer to lead our recommendation
system team. Required skills: Python, PyTorch, distributed training,
MLOps, Kubernetes. Nice to have: Rust, Ray. 5+ years experience.""",
        meta={"source": "job_board", "id": "J001"}),
    
    Document(
        content="""Data Scientist — HealthAI Inc.
Location: Remote (US only)
Compensation: $140K-$170K + equity
Join our healthcare analytics team. You'll build predictive models for
patient outcomes. Must have: Python, R, SQL, scikit-learn, statistics.
Experience with HIPAA compliance preferred. 3+ years in data science.""",
        meta={"source": "job_board", "id": "J002"}),
    
    Document(
        content="""Full Stack Developer at FinanceFlow
Based in: New York, NY (On-site)
Pay range: $150,000 to $190,000 per year
Build and maintain our trading platform dashboard. Tech stack:
React, TypeScript, Node.js, PostgreSQL, Redis, AWS.
Must be comfortable with high-frequency data streams.
2-4 years experience minimum.""",
        meta={"source": "job_board", "id": "J003"}),
    
    Document(
        content="""DevOps Engineer - CloudNative Solutions
Location: Austin, TX or Remote
Salary range: $130k-$165k annually
We need someone to manage our Kubernetes clusters and CI/CD pipelines.
Required: Docker, Kubernetes, Terraform, GitHub Actions, Linux.
AWS or GCP certification is a plus. 3+ years DevOps experience.""",
        meta={"source": "job_board", "id": "J004"}),
    
    Document(
        content="""NLP Research Scientist at LangLab
Location: London, UK (Hybrid, 2 days/week in office)
Package: £90,000-£120,000 + benefits
Work on cutting-edge LLM research: fine-tuning, RLHF, eval frameworks.
Essential: Python, transformers, deep learning, NLP.
Publications in top-tier venues (ACL, EMNLP, NeurIPS) expected.
PhD or equivalent research experience.""",
        meta={"source": "job_board", "id": "J005"}),
    
    Document(
        content="""Backend Engineer — StartupXYZ
Fully Remote, anywhere in the world
Salary: Competitive (based on experience and location)
Build scalable microservices for our B2B SaaS platform.
Stack: Go, gRPC, PostgreSQL, Kafka, Kubernetes.
We value clean code, testing, and documentation.
Open to junior and mid-level candidates.""",
        meta={"source": "job_board", "id": "J006"}),
]

# Store documents
doc_store = InMemoryDocumentStore()
doc_store.write_documents(job_docs)
print(f"Stored {doc_store.count_documents()} job postings")

## Step 2 — Extraction Prompt

Design a prompt that instructs the LLM to extract
structured fields and return **valid JSON**.

In [ ]:
EXTRACTION_TEMPLATE = """
Extract structured information from the job posting below.
Return a JSON object with these fields:
{
  "title": "job title",
  "company": "company name",
  "location": "location",
  "remote_policy": "on-site | hybrid | remote",
  "salary_min": number or null,
  "salary_max": number or null,
  "currency": "USD" or "GBP" etc,
  "required_skills": ["skill1", "skill2"],
  "experience_years": number or null
}

Rules:
- Extract salary as annual numbers (e.g. $180K = 180000)
- If salary is not specified, use null
- Return ONLY the JSON, no extra text

Job Posting:
{{ document }}

JSON:
"""

print("Extraction template ready")

## Step 3 — Build the Extraction Pipeline

In [ ]:
retriever = InMemoryBM25Retriever(document_store=doc_store, top_k=3)
prompt_builder = PromptBuilder(template=EXTRACTION_TEMPLATE)
generator = OllamaChatGenerator(
    model="qwen3.5:9b",
    url="http://localhost:11434",
    generation_kwargs={"temperature": 0, "num_predict": 512}
)

def safe_parse_json(text: str) -> dict:
    """Extract JSON from LLM response, handling markdown blocks."""
    text = text.strip()
    # Remove markdown code blocks
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0]
    elif "```" in text:
        text = text.split("```")[1].split("```")[0]
    # Remove thinking tags
    if "<think>" in text:
        text = text.split("</think>")[-1]
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Try to find JSON object in the text
        start = text.find("{")
        end = text.rfind("}") + 1
        if start >= 0 and end > start:
            try:
                return json.loads(text[start:end])
            except json.JSONDecodeError:
                pass
        return {"error": "Failed to parse JSON", "raw": text[:200]}

print("Pipeline components ready")

## Step 4 — Run Extraction

Search for job postings and extract structured data.

In [ ]:
def search_and_extract(query: str) -> list:
    """Search for jobs and extract structured data from each result."""
    print(f"\n{'='*60}")
    print(f"Search: {query}")
    print(f"{'='*60}")
    
    # Step 1: Retrieve matching documents
    results = retriever.run(query=query)
    docs = results["documents"]
    print(f"Found {len(docs)} matching postings\n")
    
    extractions = []
    for i, doc in enumerate(docs):
        print(f"--- Extracting from document {i+1} (ID: {doc.meta.get('id', '?')}) ---")
        
        # Step 2: Build extraction prompt
        prompt_result = prompt_builder.run(document=doc.content)
        prompt_text = prompt_result["prompt"]
        
        # Step 3: Generate extraction
        messages = [ChatMessage.from_user(prompt_text)]
        gen_result = generator.run(messages=messages)
        raw_reply = gen_result["replies"][0].text
        
        # Step 4: Parse JSON
        extracted = safe_parse_json(raw_reply)
        extracted["source_id"] = doc.meta.get("id", "unknown")
        extractions.append(extracted)
        
        print(f"  Title: {extracted.get('title', 'N/A')}")
        print(f"  Company: {extracted.get('company', 'N/A')}")
        print(f"  Salary: {extracted.get('salary_min', '?')}-{extracted.get('salary_max', '?')}")
        print()
    
    return extractions

print("Search & extract function ready")

In [ ]:
# Search 1: ML/AI roles
ml_jobs = search_and_extract("machine learning data science AI")

In [ ]:
# Search 2: Remote roles
remote_jobs = search_and_extract("remote engineer developer")

## Step 5 — Aggregate & Analyze Extractions

In [ ]:
# Extract ALL jobs
print("Extracting all job postings...")
all_extractions = []

for doc in job_docs:
    prompt_result = prompt_builder.run(document=doc.content)
    messages = [ChatMessage.from_user(prompt_result["prompt"])]
    gen_result = generator.run(messages=messages)
    extracted = safe_parse_json(gen_result["replies"][0].text)
    extracted["source_id"] = doc.meta.get("id", "unknown")
    all_extractions.append(extracted)

# Display as formatted JSON
print(f"\nExtracted {len(all_extractions)} job postings:\n")
print(json.dumps(all_extractions, indent=2))

In [ ]:
# Analyze the extracted data
print("\n📊 Aggregate Analysis")
print("=" * 50)

# Salary analysis (USD only)
salaries = [(e.get("title", "?"), e.get("salary_min"), e.get("salary_max"))
            for e in all_extractions
            if e.get("salary_min") and e.get("currency") == "USD"]

if salaries:
    print("\nSalary Ranges (USD):")
    for title, lo, hi in sorted(salaries, key=lambda x: x[2] or 0, reverse=True):
        print(f"  {title}: ${lo:,} - ${hi:,}")

# Skills frequency
from collections import Counter
all_skills = []
for e in all_extractions:
    skills = e.get("required_skills", [])
    if isinstance(skills, list):
        all_skills.extend(s.lower() for s in skills)

print("\nTop Skills:")
for skill, count in Counter(all_skills).most_common(10):
    print(f"  {skill}: {count}")

# Remote policy
policies = Counter(e.get("remote_policy", "unknown") for e in all_extractions)
print("\nRemote Policies:")
for policy, count in policies.most_common():
    print(f"  {policy}: {count}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **BM25 retrieval** | `InMemoryBM25Retriever` to find matching docs |
| **Structured extraction** | Prompt template → LLM → JSON |
| **JSON parsing** | Robust parser handles code blocks, errors |
| **Aggregation** | Analyze extracted fields across documents |

## Pipeline Architecture
```
User Query
    ↓
BM25 Retriever → Top-K documents
    ↓
For each doc:
    PromptBuilder → Extraction prompt
        ↓
    OllamaChatGenerator → Raw JSON text
        ↓
    JSON Parser → Structured dict
    ↓
Aggregate → Analysis / Export
```

## Extending This Notebook
- Add Pydantic models for strict field validation
- Export to CSV/parquet for downstream analytics
- Chain multiple extractors (NER → Relations → Summary)
- Use embedding retriever for semantic search